# Part A __ Load and inspect :-

In [ ]:
import pandas as pd
students = pd.read_csv(
 "data/raw/students_messy.csv",
 na_values=["", "missing", "N/A", "-"]
)
print(students.head())
print(students.shape)
print(students.dtypes)
print(students.isna().sum())
print("Exact duplicates:", students.duplicated().sum())

# Part B — Select, Filter, and Handle Duplicates :-

### B1 — Performance View

In [ ]:
performance_view = students[
    ["student_id", "name", "attendance", "assignment", "midterm", "final"]
]

display(performance_view)

### B1 — Select Likely AI Students

In [ ]:
ai_students = students[
    students["program"].str.contains("AI", case=False, na=False)
]

display(ai_students[["student_id", "name", "program"]])

### B1 — Students Below 75% Attendance

In [ ]:
below_75 = students[students["attendance"] < 75]

display(below_75)

### B1 — Final Marks At Least 80

In [ ]:
final_80 = students[students["final"] >= 80]

display(final_80)

### B1 — Sort by Final Marks

In [ ]:
sorted_performance = students.sort_values(
    by="final",
    ascending=False,
    na_position="last"
)

display(
    sorted_performance[
        ["student_id", "name", "attendance", "assignment", "midterm", "final"]
    ]
)

### B1 — Combined Series Conditions

In [ ]:
combined_filter = students[
    (students["attendance"] < 75) &
    (students["final"] >= 50)
]

display(combined_filter[["student_id", "name", "attendance", "final"]])

### B1 — loc and iloc Selection

In [ ]:
# Using .loc
loc_selection = students.loc[
    students["final"] >= 80,
    ["student_id", "name", "final"]
]

print("Using .loc:")
display(loc_selection)

# Using .iloc with equivalent row positions and column positions
row_positions = students.index[students["final"] >= 80]
column_positions = [
    students.columns.get_loc("student_id"),
    students.columns.get_loc("name"),
    students.columns.get_loc("final")
]

iloc_selection = students.iloc[row_positions, column_positions]

print("Using .iloc:")
display(iloc_selection)

### B2 — Investigate Exact Duplicates

In [ ]:
duplicates = students[students.duplicated(keep=False)]

print("All exact duplicate occurrences:")
display(duplicates)

print("Rows to remove:", students.duplicated().sum())

### B2 — Remove Exact Duplicates

In [ ]:
students = students.drop_duplicates().reset_index(drop=True)

print("New shape:", students.shape)
print("Remaining exact duplicates:", students.duplicated().sum())

display(students)

### B2 — Investigate Repeated Student IDs

In [ ]:
repeated_ids = students[
    students["student_id"].duplicated(keep=False)
].sort_values("student_id")

display(repeated_ids)

print("Repeated student IDs:")
print(repeated_ids["student_id"].unique())

### B3 — Standardize Program and City Text

In [ ]:
students["program"] = students["program"].astype("string").str.strip()
students["city"] = students["city"].astype("string").str.strip()

print("Program data type:", students["program"].dtype)
print("City data type:", students["city"].dtype)

display(students[["program", "city"]])

### B3 — Standardize Program Labels

In [ ]:
students["program"] = students["program"].str.upper()

program_mapping = {
    "BS AI": "BSAI",
    "BS-AI": "BSAI",
    "AI": "BSAI",
    "BS DS": "BSDS"
}

students["program"] = students["program"].replace(program_mapping)

print("Standardized program labels:")
print(students["program"].unique())

display(students[["student_id", "name", "program"]])

### B3 — Standardize City Labels

In [ ]:
students["city"] = students["city"].str.title()

students["city"] = students["city"].replace({
    "Rwp": "Rawalpindi"
})

students["city"] = students["city"].fillna("Unknown")

print("Standardized city labels:")
print(students["city"].unique())

display(students[["student_id", "name", "city"]])

# Part C — Types, Ranges, and Missing Values :-

### C1 — Convert Assessment Columns to Numeric

In [ ]:
score_columns = ["attendance", "assignment", "midterm", "final"]

for column in score_columns:
    students[column] = pd.to_numeric(
        students[column],
        errors="coerce"
    )

display(students[score_columns].dtypes)

### C1 — Parse Mixed Admission Dates

In [ ]:
students["admission_date"] = pd.to_datetime(
    students["admission_date"],
    dayfirst=True,
    errors="coerce"
)

display(students[["student_id", "admission_date"]])

### C1 — Convert Student ID to String and Check Missing Values

In [ ]:
students["student_id"] = students["student_id"].astype("string")

print("Data types:")
display(students.dtypes)

print("\nMissing values:")
display(students.isna().sum())

### C2 — Identify Values Outside the 0–100 Range

In [ ]:
score_columns = ["attendance", "assignment", "midterm", "final"]

invalid_records = []

for column in score_columns:
    invalid = students[
        (students[column] < 0) | (students[column] > 100)
    ].copy()

    for _, row in invalid.iterrows():
        invalid_records.append({
            "student_id": row["student_id"],
            "column": column,
            "invalid_value": row[column]
        })

invalid_records = pd.DataFrame(invalid_records)

display(invalid_records)

### C2 — Record Invalid Values in Cleaning Log

In [ ]:
cleaning_log = pd.DataFrame(columns=[
    "student_id",
    "column",
    "original_value",
    "cleaning_decision",
    "reason"
])

for _, row in invalid_records.iterrows():
    cleaning_log.loc[len(cleaning_log)] = {
        "student_id": row["student_id"],
        "column": row["column"],
        "original_value": row["invalid_value"],
        "cleaning_decision": "Replace with missing",
        "reason": "Value is outside the valid 0–100 range"
    }

display(cleaning_log)

### C2 — Replace Invalid Values With Missing

In [ ]:
for column in score_columns:
    invalid_mask = (
        (students[column] < 0) |
        (students[column] > 100)
    )

    students.loc[invalid_mask, column] = pd.NA

print("Invalid values after cleaning:")

remaining_invalid = {}

for column in score_columns:
    remaining_invalid[column] = (
        ((students[column] < 0) | (students[column] > 100)).sum()
    )

display(pd.Series(remaining_invalid))

### C3 — Missing Values Before Imputation

In [ ]:
missing_before = students.isna().sum()

print("Missing values before imputation:")
display(missing_before)

affected_ids = students.loc[
    students[score_columns].isna().any(axis=1),
    ["student_id"] + score_columns
]

print("Students affected by missing scoring values:")
display(affected_ids)

### C3 — Create Imputation Flags

In [ ]:
for column in score_columns:
    students[f"{column}_imputed"] = students[column].isna()

display(
    students[
        ["student_id"] +
        [f"{column}_imputed" for column in score_columns]
    ]
)

### C3 — Impute Missing Attendance

In [ ]:
original_attendance_median = students["attendance"].median()

program_medians = students.groupby("program")["attendance"].transform("median")

students["attendance"] = students["attendance"].fillna(program_medians)

students["attendance"] = students["attendance"].fillna(
    original_attendance_median
)

print("Original overall attendance median:", original_attendance_median)

display(
    students[["student_id", "program", "attendance", "attendance_imputed"]]
)

### C3 — Impute Missing Assessment Marks

In [ ]:
assessment_columns = ["assignment", "midterm", "final"]

assessment_medians = {}

for column in assessment_columns:
    median_value = students[column].median()
    assessment_medians[column] = median_value
    students[column] = students[column].fillna(median_value)

print("Assessment medians used:")
display(pd.Series(assessment_medians))

### C3 — Log Imputation Rules

In [ ]:
for column in score_columns:
    if column == "attendance":
        rule = "Program median; original overall median as fallback"
    else:
        rule = "Column median"

    imputed_count = students[f"{column}_imputed"].sum()

    if imputed_count > 0:
        cleaning_log.loc[len(cleaning_log)] = {
            "student_id": "Multiple",
            "column": column,
            "original_value": "Missing",
            "cleaning_decision": rule,
            "reason": f"Imputed {imputed_count} missing value(s)"
        }

display(cleaning_log)

### C3 — Missing Values Before and After Imputation

In [ ]:
missing_after = students[score_columns].isna().sum()

missing_comparison = pd.DataFrame({
    "missing_before": missing_before[score_columns],
    "missing_after": missing_after
})

display(missing_comparison)

### C3 — Verify Scoring Columns

In [ ]:
print("Missing values in scoring columns:")
display(students[score_columns].isna().sum())

print(
    "Total missing scoring values:",
    students[score_columns].isna().sum().sum()
)

### C3 — Imputation Limitation

Median imputation is useful for completing the synthetic dataset, but it can reduce natural variation in the data and may affect calculated grades or performance analysis. The imputed values should not be treated as official marks. Real student records should be verified against the original source and handled only with appropriate authorization.

# Part D — Features, Merge, and Summaries :-

### D1 — Calculate Weighted Score

In [ ]:
students["weighted_score"] = (
    students["assignment"] * 0.20
    + students["midterm"] * 0.30
    + students["final"] * 0.50
)

display(
    students[
        ["student_id", "name", "assignment", "midterm", "final", "weighted_score"]
    ]
)

### D1 — Create Attendance Status

In [ ]:
students["attendance_status"] = pd.cut(
    students["attendance"],
    bins=[-float("inf"), 75, 85, float("inf")],
    labels=["Low", "Satisfactory", "Excellent"],
    right=False
)

display(
    students[
        ["student_id", "name", "attendance", "attendance_status"]
    ]
)

### D1 — Create Grades Using np.select()

In [ ]:
import numpy as np

conditions = [
    students["weighted_score"] >= 85,
    students["weighted_score"] >= 75,
    students["weighted_score"] >= 65,
    students["weighted_score"] >= 50
]

choices = ["A", "B", "C", "D"]

students["grade"] = np.select(
    conditions,
    choices,
    default="F"
)

display(
    students[
        ["student_id", "name", "weighted_score", "grade"]
    ]
)

### D1 — Identify At-Risk Students

In [ ]:
students["at_risk"] = (
    (students["attendance"] < 75)
    | (students["weighted_score"] < 50)
)

display(
    students[
        [
            "student_id",
            "name",
            "attendance",
            "weighted_score",
            "attendance_status",
            "grade",
            "at_risk"
        ]
    ]
)

### D1 — Verify Weighted Score Bounds

In [ ]:
print("Minimum weighted score:", students["weighted_score"].min())
print("Maximum weighted score:", students["weighted_score"].max())

print("\nScores outside 0–100:")
display(
    students[
        (students["weighted_score"] < 0)
        | (students["weighted_score"] > 100)
    ][["student_id", "name", "weighted_score"]]
)

### D2 — Program Summary

In [ ]:
program_summary = (
    students
    .groupby("program")
    .agg(
        student_count=("student_id", "count"),
        average_weighted_score=("weighted_score", "mean")
    )
    .reset_index()
)

program_summary["average_weighted_score"] = (
    program_summary["average_weighted_score"].round(2)
)

display(program_summary)

### D2 — Grade Distribution

In [ ]:
grade_distribution = (
    students["grade"]
    .value_counts()
    .sort_index()
    .rename_axis("grade")
    .reset_index(name="student_count")
)

display(grade_distribution)

### D2 — Verify Summary Counts

In [ ]:
program_total = program_summary["student_count"].sum()
grade_total = grade_distribution["student_count"].sum()
student_total = len(students)

print("Total students:", student_total)
print("Program summary total:", program_total)
print("Grade distribution total:", grade_total)

print("\nProgram summary matches student count:",
      program_total == student_total)

print("Grade distribution matches student count:",
      grade_total == student_total)

# Part E — Save, Report, and Commit :-

### E1 — Final Data Quality Checks

In [ ]:
score_columns = ["attendance", "assignment", "midterm", "final"]

print("Missing student IDs:", students["student_id"].isna().sum())
print("Duplicate student IDs:", students["student_id"].duplicated().sum())

print("\nValues outside 0–100:")
for column in score_columns:
    invalid_count = (
        (students[column] < 0) | (students[column] > 100)
    ).sum()
    
    print(f"{column}: {invalid_count}")

print("\nAll IDs unique and non-missing:",
      students["student_id"].notna().all()
      and students["student_id"].is_unique)

print("All scores within 0–100:",
      all(
          ((students[column] >= 0) & (students[column] <= 100)).all()
          for column in score_columns
      ))

### E1 — Save Cleaned Data and Summaries

In [ ]:
students.to_csv(
    "data/processed/students_clean.csv",
    index=False
)

program_summary.to_csv(
    "data/processed/program_summary.csv",
    index=False
)

grade_distribution.to_csv(
    "data/processed/grade_distribution.csv",
    index=False
)

invalid_records.to_csv(
    "reports/invalid_records.csv",
    index=False
)

cleaning_log.to_csv(
    "reports/cleaning_log.csv",
    index=False
)

print("All CSV files saved successfully.")

### E1 — Reload Cleaned CSV

In [ ]:
cleaned_check = pd.read_csv(
    "data/processed/students_clean.csv"
)

print("Shape:", cleaned_check.shape)

print("\nColumns:")
print(cleaned_check.columns.tolist())

print("\nMissing values:")
display(cleaned_check.isna().sum())

print("\nFirst 5 rows:")
display(cleaned_check.head())

### E1 — Final Cleaned Dataset

In [ ]:
display(students)

# Week 6 Cleaning Report | :- | ._.

## Data Cleaning Decisions :-

- Exact duplicate records were identified and removed. The duplicated record for student ID 1003 was an exact duplicate, so one copy was retained.
- Program labels were standardized by removing surrounding spaces, converting labels to uppercase, and mapping equivalent AI and Data Science labels to BSAI and BSDS.
- City labels were standardized using title case. "Rwp" was changed to "Rawalpindi", and missing cities were replaced with "Unknown".
- Attendance and assessment columns were converted to numeric values using errors="coerce".
- Admission dates were parsed using dayfirst=True and errors="coerce". The value 09-01-2025 therefore represents 9 January 2025 and should be verified against the original source if the intended date is uncertain.
- Values outside the valid 0–100 range were identified and replaced with missing values.
- Missing attendance was imputed using the student's program median, with the original overall attendance median used as a fallback.
- Missing assessment marks were imputed using the corresponding column median.
- Imputation flags were retained to identify values that were originally missing.
- Weighted scores were calculated using 20% assignment, 30% midterm, and 50% final marks.
- Attendance status, grades, and at-risk status were created from the cleaned data.

## Limitations :-

Median imputation can reduce variation and may affect calculated performance measures and grades. The dataset is synthetic for laboratory practice, so imputed values should not be treated as official student marks. Real student records should be verified against the original source and handled with appropriate authorization.

The raw CSV files were kept unchanged.